# TradeFlow AI — nb1_synthetic_generator

Generates synthetic multimodal trade documents (PDF + JSON ground truth) for olmOCR finetuning.

In [ ]:
!pip install faker reportlab pillow pdf2image

In [ ]:
import os
import json
import random
from faker import Faker
from reportlab.pdfgen import canvas
from reportlab.lib.pagesizes import A4
from reportlab.lib.units import inch

fake = Faker('id_ID')
OUTPUT_DIR = "./dataset/synthetic"
os.makedirs(OUTPUT_DIR, exist_ok=True)

def generate_invoice(idx: int):
    # Generate dummy data
    inv_number = f"INV-{fake.random_int(1000, 9999)}-{idx}"
    date = fake.date_this_year().strftime("%Y-%m-%d")
    buyer = fake.company()
    address = fake.address().replace("\n", ", ")
    total = fake.random_int(1000, 50000)
    
    # Draw PDF
    pdf_path = os.path.join(OUTPUT_DIR, f"invoice_{idx}.pdf")
    c = canvas.Canvas(pdf_path, pagesize=A4)
    c.setFont("Helvetica-Bold", 24)
    c.drawString(1*inch, 10*inch, "COMMERCIAL INVOICE")
    
    c.setFont("Helvetica", 12)
    c.drawString(1*inch, 9*inch, f"Invoice Number: {inv_number}")
    c.drawString(1*inch, 8.75*inch, f"Date: {date}")
    c.drawString(1*inch, 8.25*inch, f"Bill To: {buyer}")
    c.drawString(1*inch, 8*inch, f"{address}")
    
    c.drawString(1*inch, 6*inch, f"Total Amount: USD {total}")
    c.save()
    
    # Write ground truth JSON
    gt = {
        "document_type": "invoice",
        "invoice_number": inv_number,
        "date": date,
        "buyer_name": buyer,
        "buyer_address": address,
        "total_amount": total
    }
    with open(os.path.join(OUTPUT_DIR, f"invoice_{idx}.json"), "w") as f:
        json.dump(gt, f, indent=2)
        
    return pdf_path, gt

print("Generating 50 synthetic invoices...")
for i in range(50):
    generate_invoice(i)
print("Done!")
